In [ ]:
from psutil import virtual_memory
ram_gb = virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))

if ram_gb < 20:
  print('Not using a high-RAM runtime')
else:
  print('You are using a high-RAM runtime!')

Your runtime has 13.6 gigabytes of available RAM

Not using a high-RAM runtime


In [ ]:
#Conexión con Google Drive para trabajar con los archivos almacenados
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#Librerías importadas
import pandas as pd
import numpy as np
import re
import regex

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer

from sklearn.svm import SVC
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import VotingClassifier

from imblearn.over_sampling import SMOTE

#from analisisFraseologico import *

#from sklearn.linear_model import LogisticRegression
#from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

#from evaluate import load
#from datasets import load_dataset
from tqdm import tqdm
tqdm.pandas() # Initialize tqdm for pandas
from tqdm.notebook import tqdm as notebook_tqdm

from nltk import sent_tokenize, word_tokenize, Text
from nltk.probability import FreqDist
import nltk
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize

import joblib
import os
import gc

import shutil
import matplotlib.pyplot as plt
import seaborn as sns



[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
path = "/content/drive/MyDrive/Titulacion/DatasetsFinales/"

In [ ]:
#Cargar la caracteristicas del nuevo archvio con df_train
df_train = pd.read_json(path + "df_train2_featselect2.jsonl", orient='records', lines=True)

#df_test = pd.read_json(path + "df_test2_featselect2.jsonl", orient='records', lines=True)

In [ ]:
df_train

,id,label,transcription,transcription_processed,MeanWordLen,LexicalDiversity,MeanSentenceLen,StdevSentenceLen,DocumentLen,WordsPerText,...,irony_score,prop_NOUN,prop_VERB,prop_ADJ,rhetorical_questions,avg_depth,Flesch Score,Lexical Entropy,Syntactic Repetition,Unusual Word Frequency
0,5eef381b-7c3102ad.mp3,1,"Yo creo que ya lo dice la propia frase, es pri...","creer decir propio frase , privar , cada hacer...",4.761905,85.714286,29.000000,0.000000,136,21,...,0.708257,0.137931,0.379310,0.000000,0,2.000000,107.57,3.97,0.28,0.50
1,3db0b886-434a22fd.mp3,0,"El presidente de Estados Unidos, Barack Obama,...","presidente unidos , barack obama , prometer añ...",6.192308,100.000000,15.500000,1.500000,195,26,...,0.562723,0.322581,0.225806,0.096774,0,3.500000,84.46,4.74,0.26,0.50
2,1e3fd1a7-dfd534d8.mp3,1,El presidente Andrés Manuel López Obrador visi...,presidente andrés manuel lópez obrador visitar...,6.413793,86.206897,16.000000,5.000000,219,29,...,0.551360,0.218750,0.218750,0.187500,0,3.000000,65.54,4.66,0.16,0.63
3,2f593981-4087fa79.mp3,1,"La sedición, porque inflación puede haber en c...","sedición , inflación poder haber cualquiera mo...",6.615385,84.615385,16.500000,3.500000,210,26,...,0.650488,0.303030,0.151515,0.121212,0,1.000000,92.32,4.39,0.21,0.62
4,815c0b94-1002ecaa.mp3,0,Frenar la escalada de violencia en Gaza es una...,frenar escalada violencia gaza prioridad . min...,6.838710,96.774194,8.750000,3.897114,247,31,...,0.504903,0.257143,0.285714,0.200000,0,2.250000,61.52,4.84,0.26,0.52
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5995,3982793b-e3c0ec4a.mp3,0,Pero la oficina del presidente francés dice qu...,oficina presidente francés decir espíritu inic...,6.684211,94.736842,14.333333,7.133645,299,38,...,0.516391,0.302326,0.279070,0.186047,0,2.000000,71.74,5.18,0.21,0.55
5996,f0442261-0348093a.mp3,1,No sé cómo se dice en castellano. Está siendo ...,"saber cómo decir castellano . ser fucking , fu...",5.736842,84.210526,6.400000,2.154066,151,19,...,0.752366,0.187500,0.093750,0.062500,0,2.000000,121.33,4.05,0.41,0.62
5997,64ea7bec-c46cfd71.mp3,0,"Sí, porque Rajoy dice que la razón de que esa ...",", rajoy decir razón calificación crediticio ca...",7.435897,89.743590,11.750000,2.861381,341,39,...,0.670326,0.234043,0.297872,0.191489,0,2.750000,64.71,5.04,0.21,0.49
5998,59a76f7b-75cc8d14.mp3,0,Francisco era conocido como Jimmy entre los se...,francisco conocer jimmy seguidor deportivo . a...,6.500000,96.875000,6.500000,2.061553,248,32,...,0.786734,0.282051,0.153846,0.230769,0,0.833333,77.59,4.84,0.23,0.52


In [ ]:
#df_test

In [ ]:
#Listar todas las columnas de df_train

print(df_train.columns.tolist())


['id', 'label', 'transcription', 'transcription_processed', 'MeanWordLen', 'LexicalDiversity', 'MeanSentenceLen', 'StdevSentenceLen', 'DocumentLen', 'WordsPerText', 'SentencesPerText', 'num_words', 'num_chars', 'irony_score', 'prop_NOUN', 'prop_VERB', 'prop_ADJ', 'rhetorical_questions', 'avg_depth', 'Flesch Score', 'Lexical Entropy', 'Syntactic Repetition', 'Unusual Word Frequency']


In [ ]:
#En otro dataframe separar a partir de la columna 4 en adelande de df_train

df_train_features = df_train.iloc[:, 4:]

df_train_features


,MeanWordLen,LexicalDiversity,MeanSentenceLen,StdevSentenceLen,DocumentLen,WordsPerText,SentencesPerText,num_words,num_chars,irony_score,prop_NOUN,prop_VERB,prop_ADJ,rhetorical_questions,avg_depth,Flesch Score,Lexical Entropy,Syntactic Repetition,Unusual Word Frequency
0,4.761905,85.714286,29.000000,0.000000,136,21,1,29,136,0.708257,0.137931,0.379310,0.000000,0,2.000000,107.57,3.97,0.28,0.50
1,6.192308,100.000000,15.500000,1.500000,195,26,2,31,196,0.562723,0.322581,0.225806,0.096774,0,3.500000,84.46,4.74,0.26,0.50
2,6.413793,86.206897,16.000000,5.000000,219,29,2,32,220,0.551360,0.218750,0.218750,0.187500,0,3.000000,65.54,4.66,0.16,0.63
3,6.615385,84.615385,16.500000,3.500000,210,26,2,33,211,0.650488,0.303030,0.151515,0.121212,0,1.000000,92.32,4.39,0.21,0.62
4,6.838710,96.774194,8.750000,3.897114,247,31,4,35,250,0.504903,0.257143,0.285714,0.200000,0,2.250000,61.52,4.84,0.26,0.52
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5995,6.684211,94.736842,14.333333,7.133645,299,38,3,43,301,0.516391,0.302326,0.279070,0.186047,0,2.000000,71.74,5.18,0.21,0.55
5996,5.736842,84.210526,6.400000,2.154066,151,19,5,32,155,0.752366,0.187500,0.093750,0.062500,0,2.000000,121.33,4.05,0.41,0.62
5997,7.435897,89.743590,11.750000,2.861381,341,39,4,47,344,0.670326,0.234043,0.297872,0.191489,0,2.750000,64.71,5.04,0.21,0.49
5998,6.500000,96.875000,6.500000,2.061553,248,32,6,39,253,0.786734,0.282051,0.153846,0.230769,0,0.833333,77.59,4.84,0.23,0.52


In [ ]:
#df_test_features = df_test.iloc[:, 4:]
#df_test_features  # Asumiendo que las características están desde la columna 4

In [ ]:
#Registros de cada etiqueta
y = df_train['label']
y.value_counts()

,count
label,
0,3168
1,2832


In [ ]:
#y_test_true = df_test['label1']
#y_test.value_counts()

In [ ]:
# Modelos de clasificación tradicionales
SVC_model = LinearSVC(dual='auto')
SVM_model = SVC(kernel='linear', probability=True)
RF_model = RandomForestClassifier(n_estimators=100, random_state=42)
MLP_model = MLPClassifier(hidden_layer_sizes=(100,), max_iter=500)
XGB_model = XGBClassifier(n_estimators=100)

models = {
    'SVC': SVC_model,
    'SVM': SVM_model,
    'RF': RF_model,
    'MLP': MLP_model,
    'XGB': XGB_model,
    'VC': VotingClassifier(estimators=[('RF', RF_model),
                                       ('SVM', SVM_model), ('MLP', MLP_model),
                                       ('XGB', XGB_model)], voting='soft')
}

In [ ]:
# Este es la cantidad de n-gramas que se eligen para cada prueba.
# Define max_features values to iterate through
max_features_values = [500, 1000, 2000, 3000, 5000]

In [ ]:
all_results = []

for max_features in notebook_tqdm(max_features_values, desc="Max Features"):
  print(f"Processing with max_features = {max_features}")

  vectorizer = TfidfVectorizer(max_features=max_features, ngram_range=(1, 2)) # Se configuran unigramas y bigramas.

  tfidf_train= vectorizer.fit_transform(df_train['transcription_processed'])
  #tfidf_test = vectorizer.transform(df_test['transcription_processed'])

  #categorias_df_train = df_train['caracteristicas'].apply(pd.Series).fillna(0)
  #categorias_df_test = df_test['caracteristicas'].apply(pd.Series).fillna(0)

  tfidf_df_train = pd.DataFrame(tfidf_train.toarray())
  #tfidf_df_test = pd.DataFrame(tfidf_test.toarray())

  combined_df_train = pd.concat([tfidf_df_train, df_train_features],  axis=1)
  X = combined_df_train
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

  # Convertir todos los nombres de columna en cadenas antes de escalar
  X_train.columns = X_train.columns.astype(str)
  X_test.columns = X_test.columns.astype(str)
  #combined_df_test.columns = combined_df_test.columns.astype(str)

  # Antes de aplicar SMOTE
  imputer = SimpleImputer(strategy='mean')  #También puede usarse "mediana" u otras estrategias.
  X_train_imputed = imputer.fit_transform(X_train)
  X_test_imputed = imputer.transform(X_test)

  # Se inicializa el MinMaxScaler
  scaler = MinMaxScaler()
  # Se ajustan y transforman los datos de entrenamiento y prueba.
  #scaled_text_x_train = scaler.fit_transform(combined_df_train)
  scaled_text_x_train = scaler.fit_transform(X_train_imputed)
  scaled_text_x_test = scaler.transform(X_test_imputed)
  #scaled_text_x_test = scaler.transform(combined_df_test)

  smote = SMOTE(random_state=42)
  X_train_balanced, y_train_balanced = smote.fit_resample(scaled_text_x_train, y_train)

  del combined_df_train
  #del combined_df_test
  del tfidf_df_train
  #del tfidf_df_test
  #del df_train_features
  #del df_test_features

  del tfidf_train
  #del tfidf_test
  gc.collect()

  #X = scaled_text_x_train
  #X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

  #smote = SMOTE(random_state=42)
  #X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)


  results = []
  for name, model in notebook_tqdm(models.items(), desc="Training Models"):
      model.fit(X_train_balanced, y_train_balanced)
      #model.fit(scaled_text_x_train, y_train_balanced)

      model_name = name
      model_folder = f"/content/drive/MyDrive/Titulacion/Entrenamiento/ConLema2/{model_name}max_features{max_features}"

      # Crear la carpeta del modelo si no existe.
      os.makedirs(model_folder, exist_ok=True)
      print(f"Processing model: {model_name} with max_features = {max_features}")

      # Guardar el modelo, el vectorizador y el escalador en la carpeta del modelo.
      joblib.dump(model, os.path.join(model_folder, 'model.pkl'))
      joblib.dump(vectorizer, os.path.join(model_folder, 'vectorizer.pkl'))
      joblib.dump(scaler, os.path.join(model_folder, 'scaler.pkl'))
      joblib.dump(imputer, os.path.join(model_folder, 'imputer.pkl'))
      #print(f"Model, vectorizer, and scaler saved to: {model_folder}")

      #y_pred = model.predict(X_test)
      y_pred = model.predict(scaled_text_x_test)
      report = classification_report(y_test, y_pred, output_dict=True, zero_division='warn')
      results.append({
          'Model': name,
          'Accuracy': accuracy_score(y_test, y_pred),
          'Precision': report['macro avg']['precision'],
          'Recall': report['macro avg']['recall'],
          'F1-score': report['macro avg']['f1-score'],
          'max_features': max_features
      })
  all_results.extend(results)

Max Features:   0%|          | 0/5 [00:00<?, ?it/s]

Processing with max_features = 500


Training Models:   0%|          | 0/6 [00:00<?, ?it/s]

Processing model: SVC with max_features = 500


KeyboardInterrupt: 

In [ ]:
X

In [ ]:
results_df = pd.DataFrame(all_results)
results_df

In [ ]:
# Se ordenan los resultados por F1-Score en orden descendente y luego por accuracy.
top_10_models = results_df.sort_values(by=['F1-score', 'Accuracy'], ascending=[False, False]).head(10)

top_10_models


In [ ]:
# Se ordena el DataFrame por F1-Score orden descendente, y se seleccionan los 10 primeros.
top_10_models = results_df.sort_values(by='F1-score', ascending=False).head(10)

top_10_models


In [ ]:
# Se pretende borrar el contenido de /content/drive/MyDrive/PruebasHM2025/SinLemma/top_models/, si no está vacía

import shutil
import os

#Definición del path del directorio
dir_path = "/content/drive/MyDrive/Titulacion/Entrenamiento/modelos/modelosCS"

# Se comprueba si el directorio existe y no está vacío.
if os.path.exists(dir_path) and os.listdir(dir_path):
    # Se borra el contenido del directorio
    shutil.rmtree(dir_path)
    print(f"Directory '{dir_path}' and its contents have been deleted.")

    #Se vuelve a crear el directorio vacío
    os.makedirs(dir_path, exist_ok=True)
    print(f"Empty directory '{dir_path}' has been recreated.")

elif os.path.exists(dir_path):
    print(f"Directory '{dir_path}' exists but is empty.")

else:
    print(f"Directory '{dir_path}' does not exist.")
    os.makedirs(dir_path, exist_ok=True)
    print(f"Empty directory '{dir_path}' has been created.")

In [ ]:
# Se guardan los 5 modelos principales basados en f1_macro en un archivo en Google Drive.
top_5_models = results_df.sort_values(by='F1-score', ascending=False).head(5)

In [ ]:
top_5_models

In [ ]:
# Se guardan los 5 modelos principales en un archivo CSV en Google Drive.
top_5_models.to_csv("/content/drive/MyDrive/Titulacion/Entrenamiento/modelos/modelosCS/top_5_models.csv", index=False)


In [ ]:
# Leer top_5_models.csv

import pandas as pd

# Cargar el archivo CSV en un DataFrame de pandas.
top_5_models = pd.read_csv("/content/drive/MyDrive/Titulacion/Entrenamiento/modelos/modelosCS/top_5_models.csv")

# Visualizar los 5 mejores modelos.
top_5_models




---


##HASTA AHÍ PORQUE YA SE LOS TRES MEJORES MODELOS

In [ ]:
# Leer top_5_models.csv

# Se carga el CSV
top_5_models = pd.read_csv("/content/drive/MyDrive/Titulacion/Entrenamiento/modelos/modelosCS/top_5_models.csv")

# Se visualizan los mejores 5 modelos
top_5_models

In [ ]:
path = "/content/drive/MyDrive/Titulacion/DatasetsFinales/"

In [ ]:
df_test = pd.read_json(path + "df_train2_featselect2.jsonl", orient='records', lines=True)

In [ ]:
#ESTE OK PARA PREDECIR CON NUEVO TEST
# Predecir con los modelos almacendos segun top_5_models

import pandas as pd
import joblib
import numpy as np
from sklearn.metrics import classification_report, accuracy_score
import shutil
import os


#AQUÍ DEBO CARGAR EL TEST QUE QUIERO PREDECIR

path = "/content/drive/MyDrive/Titulacion/DatasetsFinales/"

# Cargar los datos de prueba
#df_test = pd.read_json(path + "df_test_feat.jsonl", orient='records', lines=True)
df_test = pd.read_json(path + "df_train2_featselect2.jsonl", orient='records', lines=True)
#df_test = pd.read_json(path + "df_test2_featselect2.jsonl", orient='records', lines=True)
df_test

In [ ]:
from sklearn.model_selection import train_test_split
df_test_features = df_test.iloc[:, 4:] # Suponiendo que las características comienzan a partir de la quinta columna

model_folder = "/content/drive/MyDrive/Titulacion/Entrenamiento/ConLema2"

df_train, df_test = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)
all_results = []
results = []
#predictions = []

for index, row in top_5_models.iterrows():
    model_name = row['Model']
    max_features = row['max_features']
    model_folder = f"/content/drive/MyDrive/Titulacion/Entrenamiento/ConLema2/{model_name}max_features{max_features}"

    model_path = f"/content/drive/MyDrive/Titulacion/Entrenamiento/ConLema2/{model_name}max_features{max_features}/model.pkl"
    vectorizer_path = f"/content/drive/MyDrive/Titulacion/Entrenamiento/ConLema2/{model_name}max_features{max_features}/vectorizer.pkl"
    scaler_path = f"/content/drive/MyDrive/Titulacion/Entrenamiento/ConLema2/{model_name}max_features{max_features}/scaler.pkl"
    imputer_path = f"/content/drive/MyDrive/Titulacion/Entrenamiento/ConLema2/{model_name}max_features{max_features}/imputer.pkl"

    #Intentar:
    # Cargar el modelo, el vectorizador y el escalador.
    model = joblib.load(model_path)
    vectorizer = joblib.load(vectorizer_path)
    scaler = joblib.load(scaler_path)
    imputer = joblib.load(imputer_path)

    # Preprocesar los datos de prueba (igual que el entrenamiento)
    tfidf_test = vectorizer.transform(df_test['transcription_processed']) #Suponiendo que exista la columna «transcription_processed»
    tfidf_df_test = pd.DataFrame(tfidf_test.toarray())
    combined_df_test = pd.concat([tfidf_df_test, df_test_features], axis=1)
    combined_df_test.columns = combined_df_test.columns.astype(str)

    # Imputar valores perdidos (por ejemplo, sustituir NaN por 0).
    #combined_df_test = combined_df_test.fillna(0)  # Reemplazar NaN con 0
    X_test_imputed = imputer.transform(combined_df_test)

    scaled_text_x_test = scaler.transform(X_test_imputed)

    # Realizar predicciones
    print(f"Processing model: {model_name} with max_features = {max_features}")
    y_pred = model.predict(scaled_text_x_test)
    #prediction_label = "satire" if y_pred == 1 else "no-satire"
    prediction_counts = pd.Series(y_pred).value_counts().to_dict()
    #predictions.append(y_pred)


    #Genera la salida del Workshop
    output_df = pd.DataFrame(columns=["id", "task_1", "task_2"])
    output_df["id"] = df_test["id"].str.replace ('.mp3', '', regex = False)
    output_df["task_1"] = y_pred
    output_df["task_2"] = y_pred
    output_df["task_1"] = output_df["task_1"].map({0: 'no-satire', 1: 'satire'})
    output_df["task_2"] = output_df["task_2"].map({0: 'no-satire', 1: 'satire'})
    print (output_df)
    output_df.to_csv (os.path.join(model_folder, 'results.csv'), index = False)

    report = classification_report(df_test['label'], y_pred, output_dict=True, zero_division='warn', digits=6)

    print(classification_report(df_test['label'], y_pred, output_dict=True, zero_division='warn', digits=6))

    cm = confusion_matrix(df_test['label'], y_pred)

    #Traza la matriz de confusión.
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=['Not Satire', 'Satire'], yticklabels=['Not Satire', 'Satire'])
    plt.title(f"Confusion Matrix for {model_name, max_features}")
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.show()

    results.append({
        'Model': model_name,
        'Accuracy': accuracy_score(df_test['label'], y_pred),
        'Recall': report['macro avg']['recall'],
        'F1-score': report['macro avg']['f1-score'],
        'max_features': max_features,
        'prediction': prediction_counts,
    })

    #del combined_df_train
    del combined_df_test
    #del tfidf_df_train
    del tfidf_df_test
    #del tfidf_train
    del tfidf_test
    gc.collect()


    # Evaluar el modelo (opcional)
    #y_true = df_test["label"] # Reemplazar «label» por la columna de destino correcta.
    #report = classification_report(y_true, y_pred, output_dict=True, zero_division='warn')
    #accuracy = accuracy_score(y_true, y_pred)
    #print(f"Model {model_name} - Accuracy: {accuracy}")
    #print(report)

    #except FileNotFoundError:
    #    print(f"Model files not found for {model_name} with max_features {max_features}")
        #predictions.append(np.array([])) # or handle differently

all_results.extend(results)
results_df = pd.DataFrame(all_results).sort_values(by='F1-score', ascending=False)
top_5_models = results_df.sort_values(by='F1-score', ascending=False).head(5)
results_df



In [ ]:
#ESTE OK PARA PREDECIR CON NUEVO TEST
#Predecir con los modelos almacendos segun top_5_models

import pandas as pd
import joblib
import numpy as np
from sklearn.metrics import classification_report, accuracy_score
import shutil
import os


#AQUÍ DEBO CARGAR EL TEST QUE QUIERO PREDECIR

path = "/content/drive/MyDrive/Titulacion/DatasetsFinales/"

# Se cargan los datos de prueba
#df_test = pd.read_json(path + "df_test_feat.jsonl", orient='records', lines=True)
df_test = pd.read_json(path + "df_train2_featselect2.jsonl", orient='records', lines=True)
#df_test = pd.read_json(path + "df_test2_featselect2.jsonl", orient='records', lines=True)
df_test
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt
import joblib
import pandas as pd
import gc
import os

evaluation_results = []

for max_features in notebook_tqdm(max_features_values, desc="Evaluating max_features"):
    for model_name in models.keys():
        print(f"Evaluating model: {model_name} with max_features = {max_features}")

        model_folder = f"/content/drive/MyDrive/Titulacion/Entrenamiento/ConLema2/{model_name}max_features{max_features}"

        try:
            model = joblib.load(os.path.join(model_folder, 'model.pkl'))
            vectorizer = joblib.load(os.path.join(model_folder, 'vectorizer.pkl'))
            scaler = joblib.load(os.path.join(model_folder, 'scaler.pkl'))
            imputer = joblib.load(os.path.join(model_folder, 'imputer.pkl'))
        except FileNotFoundError:
            print(f"⚠️ Archivos no encontrados para {model_name} con max_features={max_features}")
            continue

        # Preprocesar solo el test set
        tfidf_test = vectorizer.transform(df_test['transcription_processed'])
        tfidf_df_test = pd.DataFrame(tfidf_test.toarray())
        combined_df_test = pd.concat([tfidf_df_test, df_test_features], axis=1)
        combined_df_test.columns = combined_df_test.columns.astype(str)

        # Imputar y escalar el test set
        X_test_imputed = imputer.transform(combined_df_test)
        scaled_text_x_test = scaler.transform(X_test_imputed)

        # Etiquetas reales
        y_test = df_test['label']

        # Predicción
        y_pred = model.predict(scaled_text_x_test)

        # Reporte de métricas
        report = classification_report(y_test, y_pred, output_dict=True, zero_division='warn')
        print(classification_report(y_test, y_pred, zero_division='warn'))

        # Matriz de confusión
        cm = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(6, 5))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=['No Satire', 'Satire'],
                    yticklabels=['No Satire', 'Satire'])
        plt.title(f"Confusion Matrix: {model_name} (max_features={max_features})")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.show()

        # Guardar resultados
        evaluation_results.append({
            'Model': model_name,
            'Accuracy': accuracy_score(y_test, y_pred),
            'Precision': report['macro avg']['precision'],
            'Recall': report['macro avg']['recall'],
            'F1-score': report['macro avg']['f1-score'],
            'max_features': max_features
        })

        # Limpieza de memoria
        del tfidf_test, tfidf_df_test, combined_df_test
        gc.collect()

# Crear DataFrame de resultados
results_df = pd.DataFrame(evaluation_results).sort_values(by='F1-score', ascending=False)
display(results_df)



In [ ]:
import os
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
path = "/content/drive/MyDrive/Titulacion/DatasetsFinales/"
# 1. Cargar los datos de prueba (debe tenerse la ruta o adaptarla)
df_train = pd.read_json(path + "df_train2_featselect2.jsonl", orient='records', lines=True)
df_test = pd.read_csv(df_train)

# 2. Cargar las etiquetas (y_test)
# Se asume que se tiene una columna 'label' en los datos de prueba
y_test = df_test['label']  # Cambiar 'label' por el nombre real

# 3. Cargar el vectorizer, scaler e imputer de alguno de los modelos guardados
# (Se usa uno como referencia para preprocesar)
modelo_referencia = "VC"  # Puede usarse VC o XGB
max_features_ref = 2000
model_folder_ref = f"/content/drive/MyDrive/Titulacion/Entrenamiento/ConLema2/{modelo_referencia}max_features{max_features_ref}"

# Cargar componentes de preprocesamiento
vectorizer = joblib.load(os.path.join(model_folder_ref, 'vectorizer.pkl'))
scaler = joblib.load(os.path.join(model_folder_ref, 'scaler.pkl'))
imputer = joblib.load(os.path.join(model_folder_ref, 'imputer.pkl'))

# 4. Preprocesar los datos de prueba (similar al script original)
# Aplicar TF-IDF
tfidf_test = vectorizer.transform(df_test['transcription_processed'])
tfidf_df_test = pd.DataFrame(tfidf_test.toarray())

# Combinar con características adicionales (si se tienen)
# df_test_features = df_test[['feature1', 'feature2']]  # Ajustar en base a las características
# combined_df_test = pd.concat([tfidf_df_test, df_test_features], axis=1)

# Preprocesamiento completo (imputación + escalado)
X_test_imputed = imputer.transform(tfidf_df_test)  # O combined_df_test si se usaron features
scaled_text_x_test = scaler.transform(X_test_imputed)

# 5. Obtener las clases únicas
classes = y_test.unique()

# 6. Función para matriz de confusión (igual que antes)
def plot_confusion_matrix(y_true, y_pred, classes, model_name, max_features, save_path=None):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=classes,
                yticklabels=classes)
    plt.title(f'Matriz de Confusión\n{model_name} (max_features={max_features})')
    plt.ylabel('Verdaderos')
    plt.xlabel('Predichos')
    if save_path:
        plt.savefig(os.path.join(save_path, f'confusion_matrix_{model_name}.png'))
    plt.show()
    plt.close()

# 7. Procesar los modelos (cambiar la ruta según la estructura)
modelos_a_evaluar = [
    {'Model': 'VC', 'max_features': 2000},
    {'Model': 'XGB', 'max_features': 2000}
    # Agrega otros modelos de necesiatarse
]

for modelo in modelos_a_evaluar:
    model_name = modelo['Model']
    max_features = modelo['max_features']
    model_folder = f"/content/drive/MyDrive/Titulacion/Entrenamiento/ConLema2/{model_name}max_features{max_features}"

    try:
        model = joblib.load(os.path.join(model_folder, 'model.pkl'))
        y_pred = model.predict(scaled_text_x_test)

        print(f"\nEvaluando {model_name} (max_features={max_features})")
        print(classification_report(y_test, y_pred, target_names=classes))

        plot_confusion_matrix(y_test, y_pred, classes, model_name, max_features, model_folder)

    except Exception as e:
        print(f"Error evaluando {model_name}: {str(e)}")

#LO QUE SIGUE ES UN EJEMPLO DE PREDICCION CARGANDO PRIMERO LOS MODELOS GUARDADOS (toca adecuarlo)